# 🤖 نظام أسئلة وأجوبة بالـ RAG (Retrieval-Augmented Generation Q&A)
### مشروع D1 — مسار الذكاء الاصطناعي التوليدي (GenAI Track) ⭐

---
## 🎯 المشكلة التجارية (Business Problem)
شركة عندها قاعدة معرفة (سياسات شحن، إرجاع، دفع، ضمان...) والعملاء بيسألوا نفس الأسئلة طول الوقت.
عايزين **مساعد ذكي يجاوب على أسئلة العملاء** بالاعتماد على **مستندات الشركة الحقيقية** —
مش من "مخيلة" النموذج (عشان ما يخترعش معلومات غلط / Hallucination).

**الحل:** RAG = **استرجاع (Retrieval)** المستند المناسب + **توليد (Generation)** إجابة مبنية عليه.

## 📦 ما الذي يثبته المشروع
تقطيع المستندات (Chunking) · **التضمينات (Embeddings)** · **البحث المتجهي (Vector Search)** ·
خط أنابيب RAG كامل · هندسة الـ Prompt · تقييم الاسترجاع (Hit@k) · مسار الإنتاج (LLM API).


## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

In [ ]:
# 🚀 إعداد تلقائي — Colab / Kaggle / محلي (no-op محلياً)
import os, sys, subprocess, importlib.util
REPO    = "https://github.com/ahmedelgendy11/AI-and-data-science-portfolio"
PROJECT = "genai/d1_rag_qa"          # مسار المشروع داخل portfolio/
PKGS    = ["anthropic"]          # مكتبات المشروع (تتثبّت لو ناقصة)
for _pkg in PKGS:
    if importlib.util.find_spec(_pkg.replace("-", "_")) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg])
if not os.path.isdir("data"):                 # سحابياً: نجيب الريبو ونروح لمجلد المشروع
    _clone = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(_clone):
        subprocess.run(["git", "clone", "-q", REPO + ".git"])
    os.chdir(os.path.join(_clone, "portfolio", PROJECT))
print("✓ جاهز —", os.getcwd())

## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

## 📚 قبل ما تبدأ — محتاج تذاكر إيه
| المفهوم | المصدر | بيُستخدم في إيه |
|---|---|---|
| التضمينات (Embeddings) | Jurafsky — *Speech & Language Processing* (ch.6) | تحويل النص لمتجه يمثّل المعنى |
| تشابه جيب التمام (Cosine Similarity) | Jurafsky (ch.6) | قياس قرب السؤال من المستندات |
| **RAG (Retrieval-Augmented Generation)** | Lewis et al. 2020 / وثائق LangChain | ربط الـ LLM بمصدر معرفة موثوق |
| Chunking (تقطيع المستندات) | أدلة RAG العملية | المستندات الطويلة تتقسّم لقطع قابلة للاسترجاع |
| هندسة الـ Prompt (Context injection) | وثائق OpenAI/Anthropic | حقن السياق المسترجَع في تعليمات النموذج |
| تقييم الاسترجاع (Recall@k / Hit@k) | أدبيات استرجاع المعلومات | قياس: هل جبنا المستند الصح؟ |

> 🎯 **بيُستخدم في الواقع:** شات بوتس خدمة العملاء، مساعدي الوثائق الداخلية، البحث القانوني/الطبي، كل تطبيقات "اسأل مستنداتك".
> 🛠️ **هنا أوفلاين:** نستخدم TF-IDF كـ embeddings ومولّد بسيط — والكود فيه خلايا اختيارية لـ `sentence-transformers` و LLM API حقيقي للإنتاج.


## 0️⃣ المكتبات وتحميل قاعدة المعرفة

In [ ]:
import json, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
# TODO: حمّل data/knowledge_base.json
KB = ...
print(len(KB), 'documents')

## 1️⃣ التقطيع (Chunking)
المستندات هنا قصيرة فكل مستند = قطعة واحدة. في الواقع المستندات الطويلة تتقسّم لقطع متداخلة
(مثلاً 200 كلمة بتداخل 40) عشان الاسترجاع يبقى دقيق. الدالة دي بتوضّح الفكرة.

In [ ]:
def chunk(text, size=60, overlap=15):
    # TODO: قسّم النص لقطع بطول size وتداخل overlap
    ...
chunks, meta = [], []
for doc in KB:
    for c in chunk(doc['text']):
        chunks.append(c); meta.append({'id': doc['id'], 'title': doc['title']})
print(len(chunks), 'chunks')

## 2️⃣ بناء فهرس التضمينات (Embedding Index)
نحوّل كل قطعة لمتجه TF-IDF. (في الإنتاج: متجهات كثيفة من نموذج embeddings + قاعدة متجهات زي FAISS).

In [ ]:
# TODO: درّب TfidfVectorizer على القطع واحصل على doc_vectors
embedder = ...
doc_vectors = ...
print(doc_vectors.shape)

## 3️⃣ دالة الاسترجاع (Retrieval — Top-k)

In [ ]:
def retrieve(query, k=3):
    # TODO: حوّل السؤال لمتجه، احسب cosine_similarity مع doc_vectors، رجّع أعلى k
    ...
for r in retrieve("How long does delivery take?"):
    print(r[1])

## 4️⃣ خط أنابيب RAG: استرجاع → بناء Prompt → توليد
المولّد هنا **بسيط (extractive)** — بيبني الإجابة من السياق المسترجَع. الأهم: شكل الـ **Prompt**
اللي بيتبعت للـ LLM في الإنتاج (تعليمات + سياق + سؤال) — وده موضّح في الدالة.

In [ ]:
def build_prompt(query, contexts):
    # TODO: ابنِ prompt فيه تعليمات + السياق المسترجَع + السؤال
    ...
def rag_answer(query, k=3):
    contexts = retrieve(query, k)
    prompt = build_prompt(query, contexts)
    answer, source = contexts[0][2], contexts[0][1]
    return answer, source, prompt
print(rag_answer("Can I get a refund?")[0])

## 5️⃣ (اختياري — إنتاج) توليد بـ LLM حقيقي
لو عندك مفتاح API، الخلية دي بتبعت السياق المسترجَع لـ Claude/GPT ليصيغ إجابة طبيعية.
محاطة بـ try/except عشان ما تكسرش النوتبوك أوفلاين.

In [ ]:
import os
def rag_answer_llm(query, k=3):
    contexts = retrieve(query, k)
    prompt = build_prompt(query, contexts)
    try:
        from anthropic import Anthropic                     # pip install anthropic
        client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
        msg = client.messages.create(model="claude-3-5-haiku-latest",
                                      max_tokens=300,
                                      messages=[{"role": "user", "content": prompt}])
        return msg.content[0].text
    except Exception as e:
        return f"[LLM skipped: {type(e).__name__}] Falling back to extractive answer:\n{contexts[0][2]}"

print(rag_answer_llm("Do you ship internationally and how long?"))

## 6️⃣ تقييم الاسترجاع (Retrieval Evaluation — Hit@k) 📊
المقياس الأساسي في RAG: **هل المستند الصح ظهر ضمن أعلى k نتيجة؟** نختبر بمجموعة أسئلة معروفة إجابتها.

In [ ]:
test_set = [("How many days for standard shipping?","ship-01"),
            ("How do I return an item?","ret-01"),
            ("Which cards can I pay with?","pay-01")]
# TODO: احسب Hit@1 و Hit@3 (هل المستند المتوقّع ضمن أعلى k؟)
def hit_at_k(k): ...
print('Hit@1:', hit_at_k(1))

## 7️⃣ الخلاصة والتوصيات (Conclusion)
- **النتيجة:** خط أنابيب RAG كامل بيسترجع المستند الصح ويبني عليه الإجابة — Hit@3 عالي.
- **مكوّنات النظام:** Chunking → Embeddings → Vector Search → Prompt → Generation.
- **مكافحة الـ Hallucination:** الإجابة **مقيّدة بالسياق المسترجَع** فقط، مش معرفة النموذج العامة.
- **التوصية للإنتاج:**
  1. استبدال TF-IDF بـ `sentence-transformers` (دقة أعلى في فهم المعنى).
  2. استخدام قاعدة متجهات حقيقية (FAISS / Chroma / pgvector) للتوسّع.
  3. ربط LLM حقيقي (الخلية 5) لصياغة إجابات طبيعية.
  4. إضافة تقييم للإجابة نفسها (faithfulness) مش بس الاسترجاع.

> ✅ **اللي اتعلمته:** Embeddings، Vector Search، RAG، Prompt Engineering، وتقييم الاسترجاع.
